# Model Training — AirIQ
Day 4: Train XGBoost on feature_matrix.csv, evaluate vs persistence baseline, generate SHAP chart, save artifacts.

In [2]:
import xgboost, shap

if xgboost.__version__ == "2.1.4" and shap.__version__ == "0.46.0":
    print("✅ Compatible versions already present.")
else:
    %pip uninstall -y xgboost shap
    %pip install xgboost==2.1.4 shap==0.46.0
    print("✅ Installed compatible versions. Please restart the kernel.")

c:\Users\asing\Downloads\ET AI Hackathon\airiq\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Compatible versions already present.


In [3]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe in Jupyter too
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
print(f"XGBoost  : {xgb.__version__}")
print(f"SHAP     : {shap.__version__}")
print(f"joblib   : {joblib.__version__}")

XGBoost  : 2.1.4
SHAP     : 0.46.0
joblib   : 1.5.3


## Section 1: Pre-flight — Station Audit
Confirm which stations made it into the feature matrix and row counts per station.

In [5]:
DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
ML_DIR = os.path.join("..", "ml")
os.makedirs(ML_DIR, exist_ok=True)

FEATURE_MATRIX_PATH = os.path.join(PROCESSED_DIR, "feature_matrix.csv")

df = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=["Timestamp"])

print("=" * 55)
print("STATION AUDIT")
print("=" * 55)
station_counts = df.groupby("station_name").size().sort_values(ascending=False)
for station, count in station_counts.items():
    flag = "✅" if count > 3000 else "⚠️  LOW ROW COUNT"
    print(f"  {flag}  {station:<35} {count:>6} rows")

print(f"\nTotal stations in matrix : {df['station_name'].nunique()}")
print(f"Total rows               : {len(df):,}")
print(f"Date range               : {df['Timestamp'].min()} → {df['Timestamp'].max()}")
print(f"Nulls in pm25_next_1h    : {df['pm25_next_1h'].isnull().sum()}")

EXPECTED_STATIONS = {
    "Hombegowda Nagar", "BTM Layout", "Peenya", "Hebbal",
    "Jigani", "Jayanagar 5th Block", "Kasturi Nagar", "Silk Board"
}
found = set(df["station_name"].unique())
missing = EXPECTED_STATIONS - found
extra   = found - EXPECTED_STATIONS

if missing:
    print(f"\n⚠️  MISSING from plan   : {missing}")
if extra:
    print(f"ℹ️  EXTRA (not in plan) : {extra}")
if not missing and not extra:
    print("\n✅ All 8 expected stations present.")

STATION AUDIT
  ✅  Hombegowda Nagar                      7696 rows
  ✅  BTM Layout                            6262 rows
  ✅  Kasturi Nagar                         5825 rows
  ✅  Peenya                                5560 rows
  ✅  Jigani                                5375 rows
  ✅  Jayanagar 5th Block                   5189 rows
  ✅  Silk Board                            5038 rows

Total stations in matrix : 7
Total rows               : 40,945
Date range               : 2025-01-02 00:00:00 → 2025-12-31 22:00:00
Nulls in pm25_next_1h    : 0

⚠️  MISSING from plan   : {'Hebbal'}


## Section 2: Feature Definition & Train/Test Split
Time-ordered split — NO shuffle. Last 20% of rows by date = test set.
This mirrors real-world deployment: model is always predicting into the future.

In [6]:
# These are the exact columns the model will receive at inference time.
# Timestamp and station_name are identifiers only — NOT features.
FEATURE_COLS = [
    # Current-hour pollutants
    "PM10 (µg/m³)",
    "NO2 (µg/m³)",
    "SO2 (µg/m³)",
    "CO (mg/m³)",
    "BP (mmHg)",
    # PM2.5 lag features
    "pm25_lag_1h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    # Rolling stats
    "pm25_roll_mean_3h",
    "pm25_roll_mean_24h",
    "pm25_roll_std_3h",
    # ERA5-derived weather
    "wind_speed",
    "wind_dir",
    "temp_c",
    "rel_humidity",
    # Time features
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_rush_hour",
]
TARGET_COL = "pm25_next_1h"

# Validate all feature columns exist
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in feature matrix: {missing_cols}")
print(f"✅ All {len(FEATURE_COLS)} feature columns present.")

# Drop rows with any NaN in features or target
df_clean = df[FEATURE_COLS + [TARGET_COL, "Timestamp", "station_name"]].dropna()
print(f"Rows before dropna : {len(df):,}")
print(f"Rows after  dropna : {len(df_clean):,}")
print(f"Rows dropped       : {len(df) - len(df_clean):,}")

# Time-ordered split: sort by Timestamp, take last 20% as test
df_clean = df_clean.sort_values("Timestamp").reset_index(drop=True)
split_idx = int(len(df_clean) * 0.80)
split_date = df_clean.iloc[split_idx]["Timestamp"]

train_df = df_clean.iloc[:split_idx]
test_df  = df_clean.iloc[split_idx:]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f"\nSplit date (train/test boundary) : {split_date}")
print(f"Train rows : {len(X_train):,}  |  Test rows : {len(X_test):,}")
print(f"Train shape: {X_train.shape}   |  Test shape: {X_test.shape}")
print(f"y_train range: {y_train.min():.2f} – {y_train.max():.2f} µg/m³")
print(f"y_test  range: {y_test.min():.2f} – {y_test.max():.2f} µg/m³")

✅ All 22 feature columns present.
Rows before dropna : 40,945
Rows after  dropna : 40,945
Rows dropped       : 0

Split date (train/test boundary) : 2025-10-26 00:00:00
Train rows : 32,756  |  Test rows : 8,189
Train shape: (32756, 22)   |  Test shape: (8189, 22)
y_train range: 0.02 – 985.07 µg/m³
y_test  range: 1.50 – 740.21 µg/m³


## Section 3: Persistence Baseline
Baseline = "predict pm25_next_1h = pm25_lag_1h" (last known value).
Every real model must beat this. Judges check for this comparison.

In [7]:
# Persistence model: predict that next hour = current lag_1h value
y_baseline = test_df["pm25_lag_1h"].values

baseline_rmse = np.sqrt(mean_squared_error(y_test, y_baseline))
baseline_mae  = mean_absolute_error(y_test, y_baseline)

print("=" * 40)
print("PERSISTENCE BASELINE (test set)")
print("=" * 40)
print(f"  RMSE : {baseline_rmse:.3f} µg/m³")
print(f"  MAE  : {baseline_mae:.3f} µg/m³")
print("=" * 40)
print("XGBoost must beat these numbers.")

PERSISTENCE BASELINE (test set)
  RMSE : 17.510 µg/m³
  MAE  : 5.309 µg/m³
XGBoost must beat these numbers.


## Section 4: XGBoost Training
Hyperparameters tuned for ~40K rows tabular regression.
Early stopping watches eval RMSE — stops when no improvement for 50 rounds.
Training takes ~5–20 minutes depending on hardware.

In [8]:
# Convert to DMatrix for XGBoost native API (supports early stopping cleanly)
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURE_COLS)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURE_COLS)

params = {
    "objective"        : "reg:squarederror",
    "eval_metric"      : "rmse",
    "learning_rate"    : 0.05,
    "max_depth"        : 6,
    "min_child_weight" : 5,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,       # L1 — helps with correlated features
    "reg_lambda"       : 1.0,       # L2
    "seed"             : 42,
    "verbosity"        : 1,
}

evals_result = {}

print("Training XGBoost... (watch for early stopping)")
booster = xgb.train(
    params,
    dtrain,
    num_boost_round   = 1000,
    evals             = [(dtrain, "train"), (dtest, "eval")],
    early_stopping_rounds = 50,
    evals_result      = evals_result,
    verbose_eval      = 50,         # print every 50 rounds
)

print(f"\n✅ Training complete.")
print(f"   Best round : {booster.best_iteration}")
print(f"   Best RMSE  : {booster.best_score:.4f} µg/m³")

Training XGBoost... (watch for early stopping)
[0]	train-rmse:21.64044	eval-rmse:22.94090
[50]	train-rmse:12.95981	eval-rmse:15.71037
[100]	train-rmse:11.15075	eval-rmse:16.07103
[120]	train-rmse:10.69138	eval-rmse:16.26628

✅ Training complete.
   Best round : 70
   Best RMSE  : 15.4750 µg/m³


## Section 5: Evaluation vs Baseline
Print the full metrics card. This is what goes on the dashboard accuracy panel (Day 19).

In [9]:
# Predict on test set
y_pred = booster.predict(dtest)

model_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
model_mae  = mean_absolute_error(y_test, y_pred)
model_r2   = r2_score(y_test, y_pred)
improvement = ((baseline_rmse - model_rmse) / baseline_rmse) * 100

print("=" * 55)
print("METRICS CARD — TEST SET")
print("=" * 55)
print(f"  Metric      |  Baseline (persist) |  XGBoost")
print(f"  ------------|---------------------|----------")
print(f"  RMSE        |  {baseline_rmse:>7.3f} µg/m³    |  {model_rmse:>7.3f} µg/m³")
print(f"  MAE         |  {baseline_mae:>7.3f} µg/m³    |  {model_mae:>7.3f} µg/m³")
print(f"  R²          |  {'N/A':>7}           |  {model_r2:>7.4f}")
print(f"  Improvement |  —                   |  {improvement:>6.1f}%")
print("=" * 55)

if improvement > 0:
    print(f"\n✅ XGBoost beats persistence by {improvement:.1f}% RMSE — model is valid.")
else:
    print("\n❌ XGBoost did NOT beat baseline. Check feature leakage or data issues.")
    print("   Tip: ensure pm25_lag_1h is the LAGGED value, not current PM2.5.")

METRICS CARD — TEST SET
  Metric      |  Baseline (persist) |  XGBoost
  ------------|---------------------|----------
  RMSE        |   17.510 µg/m³    |   16.266 µg/m³
  MAE         |    5.309 µg/m³    |    6.683 µg/m³
  R²          |      N/A           |   0.4570
  Improvement |  —                   |     7.1%

✅ XGBoost beats persistence by 7.1% RMSE — model is valid.


## Section 6: Learning Curve
Plot train vs eval RMSE over boosting rounds to confirm no overfitting.

In [10]:
train_rmse = evals_result["train"]["rmse"]
eval_rmse  = evals_result["eval"]["rmse"]
rounds     = range(len(train_rmse))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rounds, train_rmse, label="Train RMSE", color="#2563eb", linewidth=1.5)
ax.plot(rounds, eval_rmse,  label="Eval RMSE",  color="#dc2626", linewidth=1.5)
ax.axvline(booster.best_iteration, color="#16a34a", linestyle="--",
           label=f"Best round ({booster.best_iteration})", linewidth=1.2)
ax.axhline(baseline_rmse, color="#9333ea", linestyle=":",
           label=f"Baseline RMSE ({baseline_rmse:.2f})", linewidth=1.2)
ax.set_xlabel("Boosting Round")
ax.set_ylabel("RMSE (µg/m³)")
ax.set_title("XGBoost Learning Curve — AirIQ PM2.5 Forecasting")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

learning_curve_path = os.path.join("..", "ml", "learning_curve.png")
plt.savefig(learning_curve_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {learning_curve_path}")

✅ Saved: ..\ml\learning_curve.png


## Section 7: SHAP Explainability
SHAP TreeExplainer is fast on XGBoost (exact, not approximate).
We sample 2,000 test rows for the plot — enough for reliable feature ranking.
Saves two charts: bar (mean |SHAP|) and beeswarm (full distribution).

In [11]:
# Sample for SHAP speed — 2000 rows is plenty for reliable rankings
shap_sample_size = min(2000, len(X_test))
X_shap = X_test.sample(n=shap_sample_size, random_state=42).reset_index(drop=True)

print(f"Computing SHAP values on {shap_sample_size} test samples...")
explainer   = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)
print("✅ SHAP values computed.")

# --- Chart 1: Bar plot (mean absolute SHAP per feature) ---
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    show=False,
    max_display=22,
    color="#2563eb",
)
plt.title("Feature Importance (Mean |SHAP|) — AirIQ", fontsize=13, pad=12)
plt.tight_layout()
bar_path = os.path.join("..", "ml", "shap_bar.png")
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {bar_path}")

# --- Chart 2: Beeswarm (used in the dashboard — more informative) ---
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="dot",
    show=False,
    max_display=22,
)
plt.title("SHAP Feature Impact — AirIQ PM2.5 Forecast", fontsize=13, pad=12)
plt.tight_layout()
shap_path = os.path.join("..", "ml", "shap_chart.png")   # this is the file FastAPI serves
plt.savefig(shap_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {shap_path}")

Computing SHAP values on 2000 test samples...
✅ SHAP values computed.
✅ Saved: ..\ml\shap_bar.png
✅ Saved: ..\ml\shap_chart.png


## Section 8: Save Model Artifact
Save the trained booster with joblib. FastAPI will load this on startup.
Also save the feature column list — backend must pass features in EXACT same order.

In [12]:
# Save XGBoost booster via joblib
model_path = os.path.join(ML_DIR, "model.joblib")
joblib.dump(booster, model_path)
print(f"✅ Model saved : {model_path}")

# Save feature column list as JSON — backend loads this to build inference vectors
feature_cols_path = os.path.join(ML_DIR, "feature_cols.json")
with open(feature_cols_path, "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"✅ Feature cols: {feature_cols_path}")

# Confirm loadability right now
loaded_booster = joblib.load(model_path)
test_dmat = xgb.DMatrix(X_test.head(1), feature_names=FEATURE_COLS)
test_pred  = loaded_booster.predict(test_dmat)
print(f"\n🔁 Load + inference smoke test:")
print(f"   Input  (pm25_lag_1h) : {X_test.head(1)['pm25_lag_1h'].values[0]:.2f} µg/m³")
print(f"   Output (predicted)   : {test_pred[0]:.2f} µg/m³")
print(f"   Actual               : {y_test.head(1).values[0]:.2f} µg/m³")
print("\n✅ Model artifact verified — safe to use in FastAPI.")

✅ Model saved : ..\ml\model.joblib
✅ Feature cols: ..\ml\feature_cols.json

🔁 Load + inference smoke test:
   Input  (pm25_lag_1h) : 15.75 µg/m³
   Output (predicted)   : 18.09 µg/m³
   Actual               : 10.00 µg/m³

✅ Model artifact verified — safe to use in FastAPI.


## Section 9: Save Metrics JSON
Persisted metrics power the accuracy card on the dashboard (Day 19).

In [13]:
metrics = {
    "model"          : "XGBoost",
    "target"         : "pm25_next_1h",
    "n_features"     : len(FEATURE_COLS),
    "train_rows"     : int(len(X_train)),
    "test_rows"      : int(len(X_test)),
    "best_round"     : int(booster.best_iteration),
    "model_rmse"     : round(float(model_rmse), 4),
    "model_mae"      : round(float(model_mae), 4),
    "model_r2"       : round(float(model_r2), 4),
    "baseline_rmse"  : round(float(baseline_rmse), 4),
    "baseline_mae"   : round(float(baseline_mae), 4),
    "improvement_pct": round(float(improvement), 2),
    "stations"       : sorted(df_clean["station_name"].unique().tolist()),
    "split_date"     : str(split_date),
}

metrics_path = os.path.join(ML_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Metrics saved: {metrics_path}")
print(json.dumps(metrics, indent=2))

✅ Metrics saved: ..\ml\metrics.json
{
  "model": "XGBoost",
  "target": "pm25_next_1h",
  "n_features": 22,
  "train_rows": 32756,
  "test_rows": 8189,
  "best_round": 70,
  "model_rmse": 16.2663,
  "model_mae": 6.6827,
  "model_r2": 0.457,
  "baseline_rmse": 17.5104,
  "baseline_mae": 5.3086,
  "improvement_pct": 7.1,
  "stations": [
    "BTM Layout",
    "Hombegowda Nagar",
    "Jayanagar 5th Block",
    "Jigani",
    "Kasturi Nagar",
    "Peenya",
    "Silk Board"
  ],
  "split_date": "2025-10-26 00:00:00"
}


## Section 10: Day 4 Summary

In [14]:
print("=" * 55)
print("DAY 4 COMPLETE — ARTIFACT MANIFEST")
print("=" * 55)
artifacts = [
    ("model.joblib",       "Trained XGBoost booster"),
    ("feature_cols.json",  "Feature column order for inference"),
    ("metrics.json",       "RMSE / MAE / R² / baseline"),
    ("shap_chart.png",     "Beeswarm SHAP plot (served by API)"),
    ("shap_bar.png",       "Bar SHAP plot"),
    ("learning_curve.png", "Train vs eval RMSE over rounds"),
]
for fname, desc in artifacts:
    fpath = os.path.join(ML_DIR, fname)
    exists = "✅" if os.path.exists(fpath) else "❌ MISSING"
    size   = f"{os.path.getsize(fpath):,} bytes" if os.path.exists(fpath) else ""
    print(f"  {exists}  {fname:<25} {desc}  {size}")

print("\n🚀 Ready for Day 5: SQLite DB setup + data seeding.")

DAY 4 COMPLETE — ARTIFACT MANIFEST
  ✅  model.joblib              Trained XGBoost booster  422,371 bytes
  ✅  feature_cols.json         Feature column order for inference  431 bytes
  ✅  metrics.json              RMSE / MAE / R² / baseline  503 bytes
  ✅  shap_chart.png            Beeswarm SHAP plot (served by API)  173,034 bytes
  ✅  shap_bar.png              Bar SHAP plot  95,625 bytes
  ✅  learning_curve.png        Train vs eval RMSE over rounds  55,646 bytes

🚀 Ready for Day 5: SQLite DB setup + data seeding.
